In [2]:
# ==============================================================================
# UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA (UNAD)
# Curso: Programación - Código: 213023
# Grupo: 213023_85
# Fase 4 - Componente práctico - Prácticas simuladas
# Ejercicio 1: Sistema Integral de Gestión de Clientes, Servicios y Reservas
# Empresa: Software FJ
# ==============================================================================
# MÓDULO 1 (Aporte Individual - Estudiante 1):
# Desarrollado por: RAFAEL ANTONIO LLERENA PEREZ
# Contenido del módulo:
# 1. Listas internas en memoria y manejo de archivo de logs (.txt)
# 2. Excepciones personalizadas del sistema
# 3. Clase abstracta general (EntidadSistema)
# 4. Clase Cliente con encapsulación y validaciones estrictas
# 5. Simulaciones 1 y 2 (Registro válido e inválido con manejo de excepciones)
# ==============================================================================

from abc import ABC, abstractmethod
from datetime import datetime

ARCHIVO_LOGS = "logs_software_fj.txt"

lista_clientes = []
lista_servicios = []
lista_reservas = []

# ==============================================================================
# 1. FUNCIÓN PARA EL MANEJO DEL ARCHIVO DE LOGS
# ==============================================================================
def registrar_log(tipo_evento, descripcion):
    try:
        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        linea = f"[{fecha_hora}] [{tipo_evento}] - {descripcion}\n"
        with open(ARCHIVO_LOGS, "a", encoding="utf-8") as archivo:
            archivo.write(linea)
    except IOError as error_io:
        print(f"Advertencia: No se pudo escribir en el archivo de logs ({error_io})")

# ==============================================================================
# 2. DEFINICIÓN DE EXCEPCIONES PERSONALIZADAS
# ==============================================================================
class SoftwareFJError(Exception):
    def __init__(self, mensaje):
        super().__init__(mensaje)

class ClienteInvalidoError(SoftwareFJError):
    pass

class ParametroFaltanteError(SoftwareFJError):
    pass

class ServicioNoDisponibleError(SoftwareFJError):
    pass

class ReservaInvalidaError(SoftwareFJError):
    pass

class CalculoInconsistenteError(SoftwareFJError):
    pass

# ==============================================================================
# 3. CLASE ABSTRACTA GENERAL (EntidadSistema)
# ==============================================================================
class EntidadSistema(ABC):
    def __init__(self, id_entidad):
        self._id_entidad = id_entidad
        self._fecha_creacion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def get_id_entidad(self):
        return self._id_entidad

    @abstractmethod
    def validar_datos(self):
        pass

    @abstractmethod
    def mostrar_informacion(self):
        pass

# ==============================================================================
# 4. CLASE CLIENTE
# ==============================================================================
class Cliente(EntidadSistema):
    def __init__(self, id_cliente, nombre, correo, telefono):
        super().__init__(id_cliente)
        self.__nombre = nombre
        self.__correo = correo
        self.__telefono = telefono
        self.validar_datos()

    def get_nombre(self):
        return self.__nombre

    def set_nombre(self, nuevo_nombre):
        if nuevo_nombre is None or str(nuevo_nombre).strip() == "":
            raise ParametroFaltanteError("El nombre del cliente es un parametro obligatorio y no puede estar vacio.")
        self.__nombre = str(nuevo_nombre).strip()

    def get_correo(self):
        return self.__correo

    def set_correo(self, nuevo_correo):
        if nuevo_correo is None or str(nuevo_correo).strip() == "":
            raise ParametroFaltanteError("El correo electronico del cliente es obligatorio.")
        correo_texto = str(nuevo_correo).strip()
        if "@" not in correo_texto or "." not in correo_texto:
            raise ClienteInvalidoError(f"El correo '{correo_texto}' no tiene un formato valido (debe contener '@' y '.').")
        self.__correo = correo_texto

    def get_telefono(self):
        return self.__telefono

    def set_telefono(self, nuevo_telefono):
        if nuevo_telefono is None or str(nuevo_telefono).strip() == "":
            raise ParametroFaltanteError("El numero de telefono del cliente es obligatorio.")
        telefono_texto = str(nuevo_telefono).strip()
        if not telefono_texto.isdigit() or len(telefono_texto) < 7:
            raise ClienteInvalidoError(f"El telefono '{telefono_texto}' es invalido (debe tener solo numeros y minimo 7 digitos).")
        self.__telefono = telefono_texto

    def validar_datos(self):
        if self._id_entidad is None or str(self._id_entidad).strip() == "":
            raise ParametroFaltanteError("El documento o ID del cliente es obligatorio.")
        self.set_nombre(self.__nombre)
        self.set_correo(self.__correo)
        self.set_telefono(self.__telefono)

    def mostrar_informacion(self):
        return f"Cliente [ID: {self._id_entidad} | Nombre: {self.__nombre} | Correo: {self.__correo} | Teléfono: {self.__telefono}]"

def registrar_cliente(id_cliente, nombre, correo, telefono):
    print(f"\n>>> Procesando solicitud de registro para el cliente: {nombre} (ID: {id_cliente})")
    try:
        for cliente_actual in lista_clientes:
            if str(cliente_actual.get_id_entidad()) == str(id_cliente).strip():
                raise ValueError(f"El ID '{id_cliente}' ya se encuentra asignado a otro cliente en la lista.")
        nuevo_cliente = Cliente(id_cliente, nombre, correo, telefono)
    except (ParametroFaltanteError, ClienteInvalidoError) as error_validacion:
        print(f"[ERROR CONTROLADO]: {error_validacion}")
        registrar_log("ERROR_VALIDACION", f"Fallo en registro de cliente ID '{id_cliente}': {error_validacion}")
        return None
    except ValueError as error_origen:
        try:
            raise ClienteInvalidoError("Operacion rechazada por duplicidad de cliente en el sistema.") from error_origen
        except ClienteInvalidoError as error_encadenado:
            print(f"[ERROR ENCADENADO]: {error_encadenado}")
            print(f" -> Causa original del error: {error_encadenado.__cause__}")
            registrar_log("ERROR_ENCADENADO", f"{error_encadenado} | Causa: {error_encadenado.__cause__}")
            return None
    else:
        lista_clientes.append(nuevo_cliente)
        print(f"[REGISTRO EXITOSO]: {nuevo_cliente.mostrar_informacion()}")
        registrar_log("EVENTO_CLIENTE", f"Registro exitoso: {nuevo_cliente.mostrar_informacion()}")
        return nuevo_cliente
    finally:
        print(f"[FIN DE OPERACIÓN]: Clientes válidos almacenados en memoria: {len(lista_clientes)}")

print("======================================================================")
print(" INICIO DE SIMULACIONES - SISTEMA SOFTWARE FJ (GRUPO 213023_85)")
print("======================================================================")
registrar_log("INICIO_SISTEMA", "Se inicia la ejecucion del Sistema de Gestion Software FJ.")

print("\n--- SIMULACIÓN 1: Registro de Cliente Válido ---")
cliente_1 = registrar_cliente("1001", "Rafael Antonio Llerena Perez", "rllerenap@unadvirtual.edu.co", "3004567890")

print("\n--- SIMULACIÓN 2A: Registro de Cliente Inválido (Correo sin formato válido) ---")
cliente_invalido_1 = registrar_cliente("1002", "Carlos Mendoza", "carlos_mendoza_correo_malo.com", "3109876543")

print("\n--- SIMULACIÓN 2B: Registro de Cliente Inválido (ID duplicado) ---")
cliente_invalido_2 = registrar_cliente("1001", "Usuario Duplicado", "duplicado@correo.com", "3201112233")


# ==============================================================================
# ESPACIOS RESERVADOS PARA EL DESARROLLO COLABORATIVO DE LOS COMPAÑEROS
# ==============================================================================

# ------------------------------------------------------------------------------
# MÓDULO 2 (ESPACIO PARA ESTUDIANTE 2): FABRICIO JOSÉ ESCOBAR OSPINO
# - Crear la Clase Abstracta 'Servicio' (que herede de EntidadSistema) con métodos abstractos para calcular costos, describir servicios y validar parámetros.
# - Crear la Clase hija 'ReservaSala' (herencia y polimorfismo).
# - Realizar las Simulaciones 3 y 4 (creación correcta e incorrecta de ReservaSala).
# ------------------------------------------------------------------------------
# (Escribir código del Estudiante 2 aquí debajo)

# Clase abstracta Servicio que hereda de EntidadSistema
class Servicio(EntidadSistema):

    def __init__(self, id_servicio, nombre, descripcion, precio_base):
        super().__init__(id_servicio)
        self._nombre = nombre
        self._descripcion = descripcion
        self._precio_base = precio_base

    # Método para validar los datos generales del servicio
    def validar_datos(self):

        if self._id_entidad is None or str(self._id_entidad).strip() == "":
            raise ParametroFaltanteError("El ID del servicio es obligatorio.")

        if self._nombre is None or str(self._nombre).strip() == "":
            raise ParametroFaltanteError("El nombre del servicio es obligatorio.")

        if self._descripcion is None or str(self._descripcion).strip() == "":
            raise ParametroFaltanteError("La descripción del servicio es obligatoria.")

        if not isinstance(self._precio_base, (int, float)) or isinstance(self._precio_base, bool) or self._precio_base <= 0:
            raise CalculoInconsistenteError("El precio base debe ser un número mayor que cero.")

        self.validar_parametros()

    # Métodos abstractos que deben implementar las clases hijas
    @abstractmethod
    def calcular_costo(self, duracion=1, impuesto=0, descuento=0):
        pass

    @abstractmethod
    def describir_servicio(self):
        pass

    @abstractmethod
    def validar_parametros(self):
        pass

    # Método auxiliar para aplicar impuestos y descuentos
    def aplicar_ajustes(self, costo_base, impuesto=0, descuento=0):

        if not isinstance(impuesto, (int, float)) or isinstance(impuesto, bool) or impuesto < 0 or impuesto > 100:
            raise CalculoInconsistenteError("El impuesto debe estar entre 0 y 100.")

        if not isinstance(descuento, (int, float)) or isinstance(descuento, bool) or descuento < 0 or descuento > 100:
            raise CalculoInconsistenteError("El descuento debe estar entre 0 y 100.")

        costo_con_descuento = costo_base - (costo_base * descuento / 100)

        costo_final = costo_con_descuento + (costo_con_descuento * impuesto / 100)

        return round(costo_final, 2)

    # Método para mostrar información general
    def mostrar_informacion(self):
        return (
            f"Servicio [ID: {self._id_entidad} | "
            f"Nombre: {self._nombre} | "
            f"Descripción: {self._descripcion} | "
            f"Precio base: ${self._precio_base}]"
        )


# Clase ReservaSala que hereda de Servicio
class ReservaSala(Servicio):

    def __init__(self, id_servicio, nombre, descripcion, precio_base, capacidad, disponible=True):

        super().__init__(id_servicio, nombre, descripcion, precio_base)

        self.__capacidad = capacidad
        self.__disponible = disponible

        self.validar_datos()

    # Validación de los parámetros específicos
    def validar_parametros(self):

        if not isinstance(self.__capacidad, int) or isinstance(self.__capacidad, bool) or self.__capacidad <= 0:
            raise ServicioNoDisponibleError("La capacidad de la sala debe ser un entero mayor que cero.")

        if not isinstance(self.__disponible, bool):
            raise ClienteInvalidoError("El estado de disponibilidad debe ser verdadero o falso.")

    # Getter de capacidad
    def get_capacidad(self):
        return self.__capacidad

    # Getter de disponibilidad
    def get_disponible(self):
        return self.__disponible

    # Setter de disponibilidad
    def set_disponible(self, estado):

        if not isinstance(estado, bool):
            raise ClienteInvalidoError("El estado de disponibilidad debe ser booleano.")

        self.__disponible = estado

    # Cálculo del costo de alquiler de la sala
    def calcular_costo(self, duracion=1, impuesto=0, descuento=0):

        if not isinstance(duracion, (int, float)) or isinstance(duracion, bool) or duracion <= 0:
            raise CalculoInconsistenteError("La duración de la reserva debe ser mayor que cero.")

        if not self.__disponible:
            raise ServicioNoDisponibleError("La sala seleccionada no se encuentra disponible.")

        costo_base = self._precio_base * duracion

        return self.aplicar_ajustes(costo_base, impuesto, descuento)

    # Descripción particular del servicio
    def describir_servicio(self):
        return (
            f"Reserva de sala: {self._nombre}. "
            f"Capacidad máxima: {self.__capacidad} personas. "
            f"Precio por hora: ${self._precio_base}."
        )

    # Información completa de la sala
    def mostrar_informacion(self):
        return (
            f"ReservaSala [ID: {self._id_entidad} | "
            f"Nombre: {self._nombre} | "
            f"Capacidad: {self.__capacidad} personas | "
            f"Precio por hora: ${self._precio_base} | "
            f"Disponible: {self.__disponible}]"
        )

# ==============================================================================
# SIMULACIONES 3 Y 4 - MÓDULO 2
# ==============================================================================

print("\n==============================================================")
print(" SIMULACIONES DEL MÓDULO 2 - RESERVA DE SALA")
print("==============================================================")


# SIMULACIÓN 3: Creación correcta de una sala
try:

    sala_1 = ReservaSala(
        "S001",
        "Sala de conferencias",
        "Sala equipada para reuniones empresariales",
        80000,
        30,
        True
    )

    lista_servicios.append(sala_1)

    print("[SIMULACIÓN 3 - EXITOSA]")
    print(sala_1.mostrar_informacion())
    print(sala_1.describir_servicio())

    registrar_log(
        "EVENTO_SERVICIO",
        f"Creación exitosa: {sala_1.mostrar_informacion()}"
    )

except SoftwareFJError as error:

    print(f"[ERROR SIMULACIÓN 3]: {error}")
    registrar_log("ERROR_SERVICIO", str(error))


# SIMULACIÓN 4: Creación incorrecta de una sala
try:

    sala_invalida = ReservaSala(
        "S002",
        "Sala pequeña",
        "Sala para reuniones",
        50000,
        0,
        True
    )

    lista_servicios.append(sala_invalida)

    print("[SIMULACIÓN 4 - EXITOSA]")
    print(sala_invalida.mostrar_informacion())

except SoftwareFJError as error:

    print("[SIMULACIÓN 4 - ERROR CONTROLADO]")
    print(f"Motivo: {error}")

    registrar_log(
        "ERROR_SERVICIO",
        f"Fallo en creación de sala: {error}"
    )

# ------------------------------------------------------------------------------
# MÓDULO 3 (ESPACIO PARA ESTUDIANTE 3): YEISON STIVEN PADILLA ZULUAGA
# - Crear las Clases hijas 'AlquilerEquipo' y 'AsesoriaEspecializada' heredando
#   de la clase abstracta 'Servicio', sobrescribiendo métodos y validando parámetros.
# - Realizar las Simulaciones 5 y 6 (creación válida e inválida de estos servicios).
# ------------------------------------------------------------------------------
# (Escribir código del Estudiante 3 aquí debajo)

class AlquilerEquipo(Servicio):
    def __init__(self, id_servicio, nombre, descripcion, precio_base, serial, deposito_seguridad=50000):
        # Llama a la clase abstracta Servicio creada por Fabricio
        super().__init__(id_servicio, nombre, descripcion, precio_base)
        self.__serial = serial
        self.__deposito_seguridad = deposito_seguridad
        self.validar_datos()

    def validar_parametros(self):
        if self.__serial is None or str(self.__serial).strip() == "":
            raise ParametroFaltanteError("El equipo debe tener un número de serie válido.")
        if not isinstance(self.__deposito_seguridad, (int, float)) or isinstance(self.__deposito_seguridad, bool) or self.__deposito_seguridad < 0:
            raise CalculoInconsistenteError("El depósito de seguridad no puede ser negativo.")

    def calcular_costo(self, duracion=1, impuesto=0, descuento=0):
        if not isinstance(duracion, (int, float)) or isinstance(duracion, bool) or duracion <= 0:
            raise CalculoInconsistenteError("La duración del alquiler debe ser mayor a cero.")

        costo_base = self._precio_base * duracion
        costo_con_ajustes = self.aplicar_ajustes(costo_base, impuesto, descuento)
        return round(costo_con_ajustes + self.__deposito_seguridad, 2)

    def describir_servicio(self):
        return f"Alquiler de Equipo: {self._nombre} [Serial: {self.__serial}]. Precio base: ${self._precio_base}."

    def mostrar_informacion(self):
        return f"AlquilerEquipo [ID: {self._id_entidad} | Nombre: {self._nombre} | Serial: {self.__serial} | Depósito: ${self.__deposito_seguridad}]"


class AsesoriaEspecializada(Servicio):
    def __init__(self, id_servicio, nombre, descripcion, precio_base, especialidad, nivel_senior=False):
        # Llama a la clase abstracta Servicio creada por Fabricio
        super().__init__(id_servicio, nombre, descripcion, precio_base)
        self.__especialidad = especialidad
        self.__nivel_senior = nivel_senior
        self.validar_datos()

    def validar_parametros(self):
        if self.__especialidad is None or str(self.__especialidad).strip() == "":
            raise ParametroFaltanteError("Debe especificar el área de especialidad de la asesoría.")
        if not isinstance(self.__nivel_senior, bool):
            raise ClienteInvalidoError("El nivel senior debe ser verdadero o falso.")

    def calcular_costo(self, duracion=1, impuesto=0, descuento=0):
        if not isinstance(duracion, (int, float)) or isinstance(duracion, bool) or duracion <= 0:
            raise CalculoInconsistenteError("La duración de la asesoría debe ser mayor a cero.")

        costo_base = self._precio_base * duracion
        if self.__nivel_senior:
            costo_base = costo_base * 1.25

        return self.aplicar_ajustes(costo_base, impuesto, descuento)

    def describir_servicio(self):
        nivel = "Senior" if self.__nivel_senior else "Estándar"
        return f"Asesoría {nivel} en {self.__especialidad}: {self._nombre}."

    def mostrar_informacion(self):
        nivel = "Senior" if self.__nivel_senior else "Estándar"
        return f"AsesoriaEspecializada [ID: {self._id_entidad} | Nombre: {self._nombre} | Especialidad: {self.__especialidad} | Nivel: {nivel}]"


print("\n==============================================================")
print(" SIMULACIONES DEL MÓDULO 3 - EQUIPOS Y ASESORÍAS")
print("==============================================================")

# SIMULACIÓN 5: Creación correcta de AlquilerEquipo y AsesoriaEspecializada
try:
    equipo_1 = AlquilerEquipo("E001", "Proyector 4K", "Proyector para eventos", 150000, "PRJ-9988", 50000)
    asesoria_1 = AsesoriaEspecializada("A001", "Consultoría en Redes", "Revisión de infraestructura", 120000, "Telecomunicaciones", True)

    lista_servicios.append(equipo_1)
    lista_servicios.append(asesoria_1)

    print("[SIMULACIÓN 5 - EXITOSA]")
    print(equipo_1.mostrar_informacion())
    print(asesoria_1.mostrar_informacion())
    registrar_log("EVENTO_SERVICIO", f"Creación exitosa: {equipo_1.mostrar_informacion()}")
    registrar_log("EVENTO_SERVICIO", f"Creación exitosa: {asesoria_1.mostrar_informacion()}")
except SoftwareFJError as error:
    print(f"[ERROR SIMULACIÓN 5]: {error}")
    registrar_log("ERROR_SERVICIO", str(error))

# SIMULACIÓN 6: Creación incorrecta de servicio (Serial vacío)
try:
    print("\n>>> Intentando crear un equipo sin número de serial...")
    equipo_invalido = AlquilerEquipo("E002", "Laptop Core i7", "Laptop de desarrollo", 80000, "", 50000)
    lista_servicios.append(equipo_invalido)
except SoftwareFJError as error:
    print(f"[SIMULACIÓN 6 - ERROR CONTROLADO]: {error}")
    registrar_log("ERROR_SERVICIO", f"Fallo en creación de equipo: {error}")


# ------------------------------------------------------------------------------
# MÓDULO 4 (ESPACIO PARA ESTUDIANTE 4):
# - Implementar métodos sobrecargados para el cálculo de costos (con impuestos,
#   descuentos o parámetros opcionales).
# - Crear la estructura principal de la Clase 'Reserva' (cliente, servicio, duración, estado).
# - Realizar las Simulaciones 7 y 8 (cálculos consistentes e inconsistentes).
# Desarrollado por: HELLER DAVID OROZCO CHARRIS
# ------------------------------------------------------------------------------
class Reserva(EntidadSistema):
    def __init__(self, id_reserva, cliente, servicio, duracion_horas):
        super().__init__(id_reserva)
        self.__cliente = cliente
        self.__servicio = servicio
        self.__duracion_horas = duracion_horas
        self.__estado = "Pendiente"
        self.validar_datos()

    def get_cliente(self):
        return self.__cliente

    def get_servicio(self):
        return self.__servicio

    def get_duracion_horas(self):
        return self.__duracion_horas

    def get_estado(self):
        return self.__estado

    def set_estado(self, nuevo_estado):
        if nuevo_estado not in ["Pendiente", "Confirmada", "Cancelada", "Procesada"]:
            raise ReservaInvalidaError("El estado de la reserva no es válido.")
        self.__estado = nuevo_estado

    def calcular_costo(self, tarifa_base, impuesto_porcentaje=0.0, descuento_monto=0.0):
        if tarifa_base <= 0:
            raise CalculoInconsistenteError(f"Tarifa base inválida ({tarifa_base}): No puede ser menor o igual a cero.")
        if self.__duracion_horas <= 0:
            raise CalculoInconsistenteError(f"Duración inválida ({self.__duracion_horas}h): Debe ser mayor a cero.")
        if descuento_monto < 0:
            raise CalculoInconsistenteError("El descuento no puede ser un valor negativo.")
        subtotal = tarifa_base * self.__duracion_horas
        valor_impuesto = subtotal * (impuesto_porcentaje / 100)
        total_parcial = subtotal + valor_impuesto - descuento_monto
        if total_parcial < 0:
            raise CalculoInconsistenteError(f"Cálculo inconsistente: El descuento aplicado (${descuento_monto}) supera el costo total con impuestos (${subtotal + valor_impuesto}).")
        return total_parcial

    def validar_datos(self):
        if self._id_entidad is None or str(self._id_entidad).strip() == "":
            raise ParametroFaltanteError("El ID de la reserva es obligatorio.")
        if self.__cliente is None:
            raise ParametroFaltanteError("La reserva debe estar asociada a un cliente válido.")
        if self.__duracion_horas <= 0:
            raise CalculoInconsistenteError("La duración de la reserva en horas debe ser mayor a cero.")

    def mostrar_informacion(self):
        return f"Reserva [ID: {self._id_entidad} | Cliente: {self.__cliente.get_nombre()} | Duración: {self.__duracion_horas}h | Estado: {self.__estado}]"

    # ==========================================================================
    # MÉTODOS AGREGADOS PARA EL MÓDULO 5 (TRABAJO COLABORATIVO)
    # ==========================================================================
    def confirmar(self):
        if self.__estado in ["Cancelada", "Procesada"]:
            raise ReservaInvalidaError(f"No se puede confirmar una reserva en estado '{self.__estado}'.")
        self.set_estado("Confirmada")
        print(f"[RESERVA CONFIRMADA]: {self.mostrar_informacion()}")
        registrar_log("EVENTO_RESERVA", f"Reserva {self._id_entidad} Confirmada.")

    def cancelar(self):
        if self.__estado == "Procesada":
            raise ReservaInvalidaError("No se puede cancelar una reserva que ya fue procesada y finalizada.")
        self.set_estado("Cancelada")
        print(f"[RESERVA CANCELADA]: {self.mostrar_informacion()}")
        registrar_log("EVENTO_RESERVA", f"Reserva {self._id_entidad} Cancelada.")

    def procesar(self):
        print(f"\n>>> Procesando la reserva: {self._id_entidad}...")
        try:
            if self.__estado != "Confirmada":
                raise ValueError(f"La reserva debe estar 'Confirmada' para procesarse (Estado actual: {self.__estado}).")
            self.set_estado("Procesada")
            print(f"[PROCESAMIENTO EXITOSO]: La reserva {self._id_entidad} ha concluido de manera exitosa.")
            registrar_log("EVENTO_RESERVA", f"Reserva {self._id_entidad} Procesada correctamente.")
        except ValueError as error_estado:
            try:
                # Encadenamiento de excepciones exigido en la guía
                raise ReservaInvalidaError("Fallo en el procesamiento por estado invalido.") from error_estado
            except ReservaInvalidaError as error_encadenado:
                print(f"[ERROR ENCADENADO AL PROCESAR]: {error_encadenado}")
                print(f" -> Causa original del error: {error_encadenado.__cause__}")
                registrar_log("ERROR_PROCESAMIENTO", f"{error_encadenado} | Causa: {error_encadenado.__cause__}")

print("\n======================================================================")
print(" INICIO DE SIMULACIONES - MÓDULO 4 (ESTUDIANTE 4: HELLER OROZCO)")
print("======================================================================")
cliente_prueba_m4 = Cliente("2001", "Heller Orozco", "hdorozcoc@unadvirtual.edu.co", "3001234567")

print("\n--- SIMULACIÓN 7: Cálculo de Costo de Reserva Exitoso (Consistente) ---")
try:
    reserva_1 = Reserva("RES-701", cliente_prueba_m4, "Servicio General de Sala", 3)
    tarifa_base_hora = 50000
    iva_porcentaje = 19.0
    descuento_aplicado = 10000
    costo_final = reserva_1.calcular_costo(tarifa_base_hora, iva_porcentaje, descuento_aplicado)
    print(f"[CÁLCULO EXITOSO]: {reserva_1.mostrar_informacion()}")
    print(f" -> Detalles: Tarifa/hora: ${tarifa_base_hora} | Horas: {3} | IVA: {iva_porcentaje}% | Descuento: ${descuento_aplicado}")
    print(f" -> Costo Total Calculado: ${costo_final:.2f}")
    registrar_log("EVENTO_RESERVA", f"Cálculo exitoso para reserva {reserva_1.get_id_entidad()}: Total ${costo_final:.2f}")
except (CalculoInconsistenteError, ParametroFaltanteError, ReservaInvalidaError) as error_mod4:
    print(f"[ERROR EN MÓDULO 4]: {error_mod4}")
    registrar_log("ERROR_MODULO_4", str(error_mod4))

print("\n--- SIMULACIÓN 8: Cálculo de Costo con Error Inconsistente ---")
try:
    reserva_2 = Reserva("RES-702", cliente_prueba_m4, "Asesoría Especializada", 2)
    tarifa_base_hora = 30000
    descuento_exagerado = 200000
    print(">>> Intentando calcular costo con descuento excesivo...")
    costo_erroneo = reserva_2.calcular_costo(tarifa_base_hora, impuesto_porcentaje=19.0, descuento_monto=descuento_exagerado)
except CalculoInconsistenteError as error_inconsistente:
    print(f"[EXCEPCIÓN CONTROLADA]: {error_inconsistente}")
    registrar_log("CALCULO_INCONSISTENTE", f"Fallo en reserva {reserva_2.get_id_entidad()}: {error_inconsistente}")
except Exception as e:
    print(f"[ERROR INESPERADO]: {e}")
finally:
    print("[FIN DE SIMULACIÓN 8]: El sistema procesó la excepción de forma estable.")

print("======================================================================")

# ------------------------------------------------------------------------------
# MÓDULO 5 (ESPACIO PARA ESTUDIANTE 5): TRABAJO COLABORATIVO EN GRUPO
# (Rafael Antonio Llerena Perez, Fabricio José Escobar Ospino,
#  Heller David Orozco Charris, Yeison Stiven Padilla Zuluaga)
# - Implementar en la Clase 'Reserva' los métodos de confirmación, cancelación y
#   procesamiento con manejo y encadenamiento de excepciones.
# - Realizar las Simulaciones 9 y 10 (reserva exitosa y reserva fallida).
# ------------------------------------------------------------------------------

print("\n======================================================================")
print(" INICIO DE SIMULACIONES - MÓDULO 5 (TRABAJO COLABORATIVO)")
print("======================================================================")

# SIMULACIÓN 9: Procesamiento de reserva exitoso
print("\n--- SIMULACIÓN 9: Procesamiento de Reserva Exitosa ---")
try:
    reserva_exitosa = Reserva("RES-901", cliente_prueba_m4, "Alquiler de Proyector", 5)
    lista_reservas.append(reserva_exitosa)
    reserva_exitosa.confirmar()
    reserva_exitosa.procesar()
except SoftwareFJError as error_reserva:
    print(f"[ERROR MÓDULO 5]: {error_reserva}")

# SIMULACIÓN 10: Procesamiento fallido (encadenamiento al intentar procesar reserva cancelada)
print("\n--- SIMULACIÓN 10: Procesamiento de Reserva Fallida (Estado Cancelada) ---")
try:
    reserva_fallida = Reserva("RES-902", cliente_prueba_m4, "Reserva Sala Juntas", 2)
    lista_reservas.append(reserva_fallida)

    # Primero la cancelamos
    reserva_fallida.cancelar()

    # Intentamos procesarla estando cancelada (esto dispara la excepción encadenada)
    reserva_fallida.procesar()
except Exception as e:
    print(f"[ERROR INESPERADO MÓDULO 5]: {e}")


# ==============================================================================
# VERIFICACIÓN DE LECTURA DEL ARCHIVO DE LOGS GENERADO
# ==============================================================================
print("\n======================================================================")
print(" CONTENIDO ACTUAL DEL ARCHIVO DE LOGS (logs_software_fj.txt)")
print("======================================================================")
try:
    with open(ARCHIVO_LOGS, "r", encoding="utf-8") as archivo_lectura:
        print(archivo_lectura.read())
except FileNotFoundError:
    print("Aun no se ha generado el archivo de logs.")
finally:
    print("======================================================================")
    print(" El sistema finalizó la ejecución de prueba de manera estable.")
    print(" (Se completaron las 10 simulaciones requeridas por la guía)")
    print("======================================================================")

 INICIO DE SIMULACIONES - SISTEMA SOFTWARE FJ (GRUPO 213023_85)

--- SIMULACIÓN 1: Registro de Cliente Válido ---

>>> Procesando solicitud de registro para el cliente: Rafael Antonio Llerena Perez (ID: 1001)
[REGISTRO EXITOSO]: Cliente [ID: 1001 | Nombre: Rafael Antonio Llerena Perez | Correo: rllerenap@unadvirtual.edu.co | Teléfono: 3004567890]
[FIN DE OPERACIÓN]: Clientes válidos almacenados en memoria: 1

--- SIMULACIÓN 2A: Registro de Cliente Inválido (Correo sin formato válido) ---

>>> Procesando solicitud de registro para el cliente: Carlos Mendoza (ID: 1002)
[ERROR CONTROLADO]: El correo 'carlos_mendoza_correo_malo.com' no tiene un formato valido (debe contener '@' y '.').
[FIN DE OPERACIÓN]: Clientes válidos almacenados en memoria: 1

--- SIMULACIÓN 2B: Registro de Cliente Inválido (ID duplicado) ---

>>> Procesando solicitud de registro para el cliente: Usuario Duplicado (ID: 1001)
[ERROR ENCADENADO]: Operacion rechazada por duplicidad de cliente en el sistema.
 -> Causa ori